In [ ]:
!pip install -q torch transformers peft bitsandbytes accelerate datasets gradio pymupdf sentencepiece
!nvidia-smi


In [ ]:
from google.colab import drive, files
import os, shutil

# Mount Google Drive to persist the fine-tuned model
drive.mount('/content/drive', force_remount=True)

# Upload the 3 PDFs from your local machine
print('Upload the 3 PDF files now...')
uploaded = files.upload()

# Backup PDFs to Drive
drive_pdf_dir = '/content/drive/MyDrive/capstone_pdfs'
os.makedirs(drive_pdf_dir, exist_ok=True)

for fname in uploaded.keys():
    if not os.path.exists(os.path.join(drive_pdf_dir, fname)):
        shutil.copy(fname, os.path.join(drive_pdf_dir, fname))

PDF_FILES = [
    'proposal.pdf',
    'Neural ODE Malware Drift Research (2).pdf',
    'Capstone_Project_Proposal .pdf',
]

print('PDFs in /content:', [f for f in PDF_FILES if os.path.exists(f)])


In [ ]:
import fitz
import re
import json
import random
import os
from difflib import SequenceMatcher

random.seed(42)

PDF_FILES = [
    'proposal.pdf',
    'Neural ODE Malware Drift Research (2).pdf',
    'Capstone_Project_Proposal .pdf',
]

TRAIN_JSONL = 'train.jsonl'
VAL_JSONL = 'val.jsonl'
MIN_PARA_CHARS = 60
VAL_FRAC = 0.10

TEMPLATE_RULES = [
    (['concept drift'], ['What is concept drift in malware detection?', 'Explain concept drift and why it degrades malware classifiers over time.']),
    (['neural ode', 'latent neural ode'], ['How does a Latent Neural ODE model malware concept drift?', 'Explain the Neural ODE approach to modeling malware feature evolution.']),
    (['neural jump ode', 'njode', 'jump'], ['What is a Neural Jump ODE (NJODE) and when is it needed?', 'How does a Neural Jump ODE handle sudden concept drift?']),
    (['bodmas'], ['What is the BODMAS dataset and why is it used?', 'Describe the BODMAS dataset for malware temporal analysis.']),
    (['tesseract'], ['What is the TESSERACT evaluation framework?', 'How does TESSERACT expose spatio-temporal bias in malware classification?']),
    (['focal loss', 'class imbalance'], ['How does Focal Loss help with class imbalance in malware detection?', 'Explain the role of Focal Loss in training drift-resilient models.']),
    (['adjoint', 'adjoint sensitivity'], ['What is the adjoint sensitivity method in Neural ODE training?', 'How does the adjoint method reduce memory usage during ODE backprop?']),
    (['encoder', 'decoder', 'latent'], ['Describe the encoder-solver-decoder pipeline of a Latent Neural ODE.', 'How does the encoder compress PE features into a latent state?']),
    (['area under time', 'aut', 'auroc'], ['What is the Area Under Time (AUT) metric?', 'How is model drift resilience measured using AUROC over time?']),
    (['stiff', 'nfe', 'function evaluation', 'dopri5'], ['What is the NFE explosion problem in Neural ODE training?', 'How do stiff ODEs affect Neural ODE solver performance?']),
    (['static classifier', 'xgboost', 'lightgbm', 'mlp', 'baseline'], ['Why do static classifiers like XGBoost and MLP fail under concept drift?', 'Compare static baselines with continuous-time models for malware detection.']),
]

FALLBACK_TEMPLATES = ['Explain the following technical content in detail:', 'Summarize the key points of this passage:', 'What does the document say about this topic?']


def extract_paragraphs_from_pdf(pdf_path):
    paragraphs = []
    doc = fitz.open(pdf_path)
    for page in doc:
        text = page.get_text()
        raw_paras = re.split(r'\n\s*\n', text)
        for para in raw_paras:
            para = para.strip()
            if para:
                paragraphs.append(para)
    doc.close()
    return paragraphs


def clean_text(text):
    text = text.replace('\u200b', '').replace('\ufeff', '').replace('\u00a0', ' ')
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    text = re.sub(r'^\s*\d{1,3}\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]{3,}', ' ', text)
    text = '\n'.join(line.rstrip() for line in text.split('\n'))
    return text.strip()


def merge_short_fragments(paragraphs, min_chars=MIN_PARA_CHARS):
    merged = []
    for p in paragraphs:
        if merged and len(merged[-1]) < min_chars:
            merged[-1] = merged[-1] + ' ' + p
        else:
            merged.append(p)
    return merged


def is_near_duplicate(a, b, threshold=0.85):
    if len(a) < 40 or len(b) < 40:
        return False
    return SequenceMatcher(None, a, b).ratio() > threshold


def deduplicate(paragraphs):
    kept = []
    for p in paragraphs:
        if not any(is_near_duplicate(p, k) for k in kept):
            kept.append(p)
    return kept


def choose_instruction(paragraph):
    low = paragraph.lower()
    for keywords, options in TEMPLATE_RULES:
        if any(kw in low for kw in keywords):
            return random.choice(options)
    return random.choice(FALLBACK_TEMPLATES)


def make_qa_pairs(paragraphs):
    pairs = []
    i = 0
    while i < len(paragraphs):
        p = paragraphs[i]
        is_header = (
            len(p) < 120
            and not p.endswith('.')
            and not p.endswith(':')
            and (p[0].isupper() or p[0].isdigit())
        )
        if is_header and i + 1 < len(paragraphs):
            clean_header = re.sub(r'\s+', ' ', p).strip()
            instruction = f"What does the document say about '{clean_header}'?"
            response = paragraphs[i + 1]
            pairs.append({'instruction': instruction, 'response': response})
            i += 2
        else:
            instruction = choose_instruction(p)
            response = p
            pairs.append({'instruction': instruction, 'response': response})
            i += 1
    return pairs


def write_jsonl(records, path):
    with open(path, 'w', encoding='utf-8') as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')


# Run pipeline
all_paragraphs = []
for pdf in PDF_FILES:
    if not os.path.exists(pdf):
        print('[WARN] File not found: ' + pdf)
        continue
    print('[INFO] Extracting: ' + pdf)
    raw = extract_paragraphs_from_pdf(pdf)
    cleaned = [clean_text(p) for p in raw if clean_text(p)]
    cleaned = [p for p in cleaned if len(p) >= 20]
    cleaned = merge_short_fragments(cleaned)
    all_paragraphs.extend(cleaned)
    print('       -> %d paragraphs extracted' % len(cleaned))

print('[INFO] Total paragraphs before dedup: %d' % len(all_paragraphs))
all_paragraphs = deduplicate(all_paragraphs)
print('[INFO] Total paragraphs after dedup: %d' % len(all_paragraphs))

pairs = make_qa_pairs(all_paragraphs)
print('[INFO] Generated %d instruction-response pairs' % len(pairs))

random.shuffle(pairs)
n_val = max(1, int(len(pairs) * VAL_FRAC))
val_pairs = pairs[:n_val]
train_pairs = pairs[n_val:]

write_jsonl(train_pairs, TRAIN_JSONL)
write_jsonl(val_pairs, VAL_JSONL)
print('[INFO] Wrote %s (%d pairs)' % (TRAIN_JSONL, len(train_pairs)))
print('[INFO] Wrote %s (%d pairs)' % (VAL_JSONL, len(val_pairs)))

# Save copies to Drive
drive_out = '/content/drive/MyDrive/capstone'
os.makedirs(drive_out, exist_ok=True)
shutil.copy(TRAIN_JSONL, os.path.join(drive_out, TRAIN_JSONL))
shutil.copy(VAL_JSONL, os.path.join(drive_out, VAL_JSONL))
print('[INFO] Copied dataset to %s' % drive_out)


In [ ]:
import os, json, random, torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE_MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
TRAIN_JSONL = 'train.jsonl'
VAL_JSONL = 'val.jsonl'
OUTPUT_DIR = './lora_adapter'
MERGED_DIR = './merged_model'
MAX_LENGTH = 512
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

MIX_WITH_GENERAL = True
GENERAL_INSTRUCTIONS = [
    ('What is the capital of France?', 'The capital of France is Paris.'),
    ('Explain photosynthesis in one sentence.', 'Photosynthesis is the process by which plants convert sunlight, carbon dioxide, and water into glucose and oxygen.'),
    ('Write a short greeting.', 'Hello! How can I help you today?'),
    ('What is 2 + 2?', '2 + 2 = 4.'),
    ('Define machine learning.', 'Machine learning is a branch of artificial intelligence where computers learn patterns from data without being explicitly programmed.'),
]

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']

USE_CUDA = torch.cuda.is_available()
print('CUDA available: ' + str(USE_CUDA))

TRAIN_ARGS = dict(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=USE_CUDA,
    logging_steps=5,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='none',
    remove_unused_columns=False,
    dataloader_pin_memory=USE_CUDA,
    seed=RANDOM_SEED,
)


def load_jsonl(path):
    records = []
    if not os.path.exists(path):
        return records
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def format_example(instruction, response):
    return f'''<|user|>
{instruction} <|assistant|>
{response} '''


def tokenize_function(examples, tokenizer):
    texts = [format_example(instr, resp) for instr, resp in zip(examples['instruction'], examples['response'])]
    tokenized = tokenizer(texts, truncation=True, max_length=MAX_LENGTH, padding='max_length')
    tokenized['labels'] = tokenized['input_ids'].copy()
    return tokenized


# Load data
train_records = load_jsonl(TRAIN_JSONL)
val_records = load_jsonl(VAL_JSONL)
print('Domain train pairs: %d' % len(train_records))
print('Domain val pairs: %d' % len(val_records))

if MIX_WITH_GENERAL and train_records:
    general = [{'instruction': i, 'response': r} for i, r in GENERAL_INSTRUCTIONS]
    mix_ratio = max(1, len(train_records) // len(general) + 1)
    general_expanded = general * mix_ratio
    random.shuffle(general_expanded)
    general_subset = general_expanded[:len(train_records)]
    train_records = train_records + general_subset
    random.shuffle(train_records)
    print('Mixed train pairs: %d' % len(train_records))


# Model & tokenizer
print('[INFO] Loading tokenizer: ' + BASE_MODEL)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if USE_CUDA:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
    )
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map='auto',
        trust_remote_code=True,
        torch_dtype=torch.float16,
    )
    model = prepare_model_for_kbit_training(model)
else:
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        device_map=None,
        trust_remote_code=True,
        torch_dtype=torch.float32,
    )
    model.to('cpu')

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


# Tokenize datasets
train_dataset = Dataset.from_list(train_records)
val_dataset = Dataset.from_list(val_records) if val_records else None


def tok(examples):
    return tokenize_function(examples, tokenizer)


train_tokenized = train_dataset.map(tok, batched=True, remove_columns=train_dataset.column_names)
val_tokenized = val_dataset.map(tok, batched=True, remove_columns=val_dataset.column_names) if val_dataset else None

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(**TRAIN_ARGS)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print('[INFO] Starting QLoRA fine-tuning...')
trainer.train()

print('[INFO] Saving LoRA adapter to ' + OUTPUT_DIR)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print('[INFO] Merging adapter into base model...')
from peft import AutoPeftModelForCausalLM
merge_kwargs = dict(trust_remote_code=True)
if USE_CUDA:
    merge_kwargs['device_map'] = 'auto'
    merge_kwargs['torch_dtype'] = torch.float16
else:
    merge_kwargs['torch_dtype'] = torch.float32

merged_model = AutoPeftModelForCausalLM.from_pretrained(OUTPUT_DIR, **merge_kwargs)
merged_model = merged_model.merge_and_unload()
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print('[INFO] Merged model saved to ' + MERGED_DIR)

# Save to Drive
drive_model_dir = '/content/drive/MyDrive/capstone/merged_model'
os.makedirs(drive_model_dir, exist_ok=True)
for f in os.listdir(MERGED_DIR):
    src = os.path.join(MERGED_DIR, f)
    dst = os.path.join(drive_model_dir, f)
    if os.path.isfile(src):
        shutil.copy(src, dst)
print('[INFO] Model copied to ' + drive_model_dir)
print('[INFO] Fine-tuning complete!')


In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import gradio as gr
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

MODEL_DIR = '/content/drive/MyDrive/capstone/merged_model'
if not os.path.exists(MODEL_DIR):
    MODEL_DIR = './merged_model'

print('[INFO] Loading model from: ' + MODEL_DIR)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    device_map='auto' if device == 'cuda' else None,
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
    trust_remote_code=True,
)
if device == 'cpu':
    model.to('cpu')
model.eval()


def generate_response(prompt, max_new_tokens=256, temperature=0.7):
    if not prompt or not prompt.strip():
        return 'Please enter a question.'
    messages = f'<|user|>\n{prompt.strip()} <|assistant|>\n'
    inputs = tokenizer(messages, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=int(max_new_tokens),
            temperature=float(temperature),
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    input_len = inputs['input_ids'].shape[1]
    generated = outputs[0][input_len:]
    response = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return response


with gr.Blocks(title='Malware Concept Drift - Fine-Tuned LLM') as demo:
    gr.Markdown('# ⚡ Malware Concept Drift - Fine-Tuned LLM')
    gr.Markdown('Ask questions about malware concept drift, Neural ODEs, NJODEs, BODMAS, TESSERACT, and related topics.')
    with gr.Row():
        with gr.Column():
            prompt = gr.Textbox(label='Your Question', placeholder='e.g. What is concept drift in malware detection?', lines=3)
            max_new_tokens = gr.Slider(minimum=32, maximum=512, value=256, step=32, label='Max New Tokens')
            temperature = gr.Slider(minimum=0.1, maximum=1.5, value=0.7, step=0.1, label='Temperature')
            submit = gr.Button('Generate', variant='primary')
        with gr.Column():
            output = gr.Textbox(label='Model Response', lines=12)
    submit.click(fn=generate_response, inputs=[prompt, max_new_tokens, temperature], outputs=output)
    gr.Examples(
        examples=[
            'What is concept drift in malware detection?',
            'How does a Neural ODE help with concept drift?',
            'What is the BODMAS dataset and why is it used?',
            'What is the TESSERACT framework?',
            'What is a Neural Jump ODE and when is it needed?',
        ],
        inputs=prompt,
    )

demo.launch(server_name='0.0.0.0', server_port=7860, share=True)
